# 04 — Hybrid Retrieval

Builds the real `EnsembleRetriever` from `retrieval.py` (vector + BM25 fused by reciprocal rank fusion) and shows both individual ranked lists side by side against the fused result, on the real corpus.

In [1]:
import sys, os
sys.path.insert(0, "..")
os.chdir("..")

from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS

from core import FAISS_DIR, get_embeddings, load_chunks, tokenize
from retrieval import BM25_K, VECTOR_K, build_ensemble_retriever

chunks = load_chunks()
embeddings = get_embeddings()
vectorstore = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)

question = "What is the difference between internal and external fragmentation?"


/var/folders/8m/z6hx0h5s18d_brpspkfpd2_w0000gn/T/ipykernel_10491/2116742115.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Vector-only ranking:

In [2]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": VECTOR_K})
vector_hits = vector_retriever.invoke(question)
for i, d in enumerate(vector_hits[:5]):
    print(f"{i+1}. {d.metadata['source_file']} / {d.metadata['location']}")


1. VIPS OS_UNIT 3_Memory organization Management .pdf / page 33
2. VIPS OS_UNIT 3_Memory organization Management .pdf / page 31
3. VIPS OS_UNIT 3_Memory organization Management .pdf / page 34
4. VIPS OS_UNIT 3_Memory organization Management .pdf / page 30
5. VIPS OS_UNIT 3_Memory organization Management .pdf / page 32


BM25-only (keyword) ranking:

In [3]:
bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
bm25_retriever.k = BM25_K
bm25_hits = bm25_retriever.invoke(question)
for i, d in enumerate(bm25_hits[:5]):
    print(f"{i+1}. {d.metadata['source_file']} / {d.metadata['location']}")


1. VIPS OS_UNIT 3_Memory organization Management .pdf / page 34
2. VIPS OS_UNIT 3_Memory organization Management .pdf / page 30
3. VIPS OS_UNIT 3_Memory organization Management .pdf / page 35
4. VIPS OS_UNIT 3_Memory organization Management .pdf / page 37
5. VIPS OS_UNIT 3_Memory organization Management .pdf / page 32


Fused (EnsembleRetriever, weighted reciprocal rank fusion) — this is what the live app actually uses:

In [4]:
ensemble = build_ensemble_retriever(vectorstore, chunks)
fused_hits = ensemble.invoke(question)
for i, d in enumerate(fused_hits[:5]):
    print(f"{i+1}. {d.metadata['source_file']} / {d.metadata['location']}")


1. VIPS OS_UNIT 3_Memory organization Management .pdf / page 34
2. VIPS OS_UNIT 3_Memory organization Management .pdf / page 30
3. VIPS OS_UNIT 3_Memory organization Management .pdf / page 35
4. VIPS OS_UNIT 3_Memory organization Management .pdf / page 32
5. VIPS OS_UNIT 3_Memory organization Management .pdf / page 37
